# NB03: DEM 기반 경사도 분석 (로봇 배송 제약 레이어)

**목적**: SRTM 30m DEM에서 성남시 경사도 계산 → Green/Yellow Zone 분류

**입력**: 
- SRTM 1-arc-second DEM: `00_data/추가/DEM_SRTM/N37E127.hgt` (또는 대체 DEM)
- `processed/seongnam_boundary.gpkg`

**출력**: 
- `processed/seongnam_dem.tif` (성남시 DEM 클립)
- `processed/seongnam_slope.tif` (경사도 래스터)
- `processed/slope_zones.gpkg` (Green/Yellow Zone 벡터)

**핵심 기준**: 
- **Green Zone** (경사도 < 5°): 로봇 배송 가능 → 드론+로봇 Full 배송
- **Yellow Zone** (경사도 ≥ 5°): 로봇 배송 제약 → 드론 스테이션에서 인간 수령

> **사전 준비**: SRTM DEM `N37E127.hgt`를 `00_data/추가/DEM_SRTM/`에 저장  
> 다운로드: USGS EarthExplorer 또는 OpenTopography에서 SRTM 1-arc-second

In [1]:
import numpy as np
import geopandas as gpd
from pathlib import Path

BASE = Path(r"C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam")
OUT = BASE / "processed"
DEM_DIR = BASE / "00_data" / "추가" / "DEM_SRTM"

# 성남시 경계 (EPSG:4326)
seongnam = gpd.read_file(OUT / "seongnam_boundary.gpkg", layer="dong")
bounds = seongnam.total_bounds  # (min_lon, min_lat, max_lon, max_lat)
print(f"성남시 Bounds (EPSG:4326): lon {bounds[0]:.4f}~{bounds[2]:.4f}, lat {bounds[1]:.4f}~{bounds[3]:.4f}")

성남시 Bounds (EPSG:4326): lon 127.0277~127.1958, lat 37.3334~37.4748


## 1. SRTM DEM 로드

SRTM .hgt 파일은 big-endian signed 16-bit integer 배열 (3601x3601 for 1-arc-second).  
각 타일은 1°x1° 범위를 커버 (N37E127 = lat 37~38, lon 127~128).

In [2]:
import rasterio
from rasterio.transform import from_bounds

# SRTM .hgt 파일 경로
hgt_path = DEM_DIR / "N37E127.hgt"

if hgt_path.exists():
    # 방법 1: rasterio로 로드
    try:
        with rasterio.open(hgt_path) as src:
            dem_full = src.read(1)
            transform = src.transform
            print(f"rasterio로 로드 성공: {dem_full.shape}, CRS: {src.crs}")
    except:
        # 방법 2: numpy 직접 로드 (rasterio 실패 시)
        dem_full = np.fromfile(hgt_path, dtype=">i2").reshape(3601, 3601)
        # SRTM N37E127: lat 37~38 (남→북), lon 127~128
        transform = from_bounds(127.0, 37.0, 128.0, 38.0, 3601, 3601)
        print(f"numpy로 로드: {dem_full.shape}")
    
    # 성남시 영역만 추출 (pixel 인덱스 계산)
    # SRTM은 상단=북쪽이므로 행 인덱스 = (38 - lat) * 3600
    row_min = int((38.0 - bounds[3]) * 3600)  # max_lat
    row_max = int((38.0 - bounds[1]) * 3600)  # min_lat
    col_min = int((bounds[0] - 127.0) * 3600) # min_lon
    col_max = int((bounds[2] - 127.0) * 3600) # max_lon
    
    # 약간의 버퍼 (10 pixels ≈ 300m)
    buf = 10
    row_min = max(0, row_min - buf)
    row_max = min(3600, row_max + buf)
    col_min = max(0, col_min - buf)
    col_max = min(3600, col_max + buf)
    
    dem = dem_full[row_min:row_max, col_min:col_max].astype(np.float32)
    
    # nodata 처리 (SRTM void = -32768)
    dem[dem < -100] = np.nan
    
    # 클리핑 영역의 좌표 정보
    clip_lon_min = 127.0 + col_min / 3600
    clip_lon_max = 127.0 + col_max / 3600
    clip_lat_min = 38.0 - row_max / 3600
    clip_lat_max = 38.0 - row_min / 3600
    
    print(f"\n성남시 DEM 클립: {dem.shape}")
    print(f"고도 범위: {np.nanmin(dem):.0f}m ~ {np.nanmax(dem):.0f}m")
    print(f"영역: lon {clip_lon_min:.4f}~{clip_lon_max:.4f}, lat {clip_lat_min:.4f}~{clip_lat_max:.4f}")
else:
    print(f"⚠ SRTM DEM 파일 없음: {hgt_path}")
    print("  → USGS EarthExplorer 또는 OpenTopography에서 N37E127.hgt 다운로드 필요")
    print("  → 또는 기존 DEM (37709.img)만으로 제한된 분석 수행")
    dem = None

⚠ SRTM DEM 파일 없음: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam\00_data\추가\DEM_SRTM\N37E127.hgt
  → USGS EarthExplorer 또는 OpenTopography에서 N37E127.hgt 다운로드 필요
  → 또는 기존 DEM (37709.img)만으로 제한된 분석 수행


## 2. 경사도 계산

경사도(slope) = arctan(√(∂z/∂x)² + (∂z/∂y)²)  
SRTM 30m 해상도에서 pixel 간격 ≈ 30m (위도 37° 기준)

In [3]:
if dem is not None:
    # 1 arc-second ≈ 30m (정확한 값: 위도 37° 기준)
    lat_center = (clip_lat_min + clip_lat_max) / 2
    dx = 30.87 * np.cos(np.radians(lat_center))  # 경도 방향 pixel 크기 (m)
    dy = 30.87  # 위도 방향 pixel 크기 (m)
    
    # 경사도 계산 (numpy gradient)
    dz_dy, dz_dx = np.gradient(dem, dy, dx)
    slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    slope_deg = np.degrees(slope_rad)
    
    print(f"경사도 범위: {np.nanmin(slope_deg):.1f}° ~ {np.nanmax(slope_deg):.1f}°")
    print(f"평균 경사도: {np.nanmean(slope_deg):.1f}°")
    
    # Zone 분류
    SLOPE_THRESHOLD = 5.0  # 도
    green_pct = np.nanmean(slope_deg < SLOPE_THRESHOLD) * 100
    yellow_pct = 100 - green_pct
    print(f"\nGreen Zone (< {SLOPE_THRESHOLD}°, 로봇 가능): {green_pct:.1f}%")
    print(f"Yellow Zone (≥ {SLOPE_THRESHOLD}°, 인간 수령): {yellow_pct:.1f}%")

## 3. 시각화 + GeoTIFF 저장

In [4]:
if dem is not None:
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    
    plt.rcParams["font.family"] = "Malgun Gothic"
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # 1) 고도 지도
    im1 = axes[0].imshow(dem, cmap="terrain",
                          extent=[clip_lon_min, clip_lon_max, clip_lat_min, clip_lat_max])
    axes[0].set_title("성남시 고도 (m)")
    plt.colorbar(im1, ax=axes[0], shrink=0.7, label="m")
    
    # 2) 경사도 지도
    im2 = axes[1].imshow(slope_deg, cmap="YlOrRd", vmin=0, vmax=30,
                          extent=[clip_lon_min, clip_lon_max, clip_lat_min, clip_lat_max])
    axes[1].set_title("경사도 (°)")
    plt.colorbar(im2, ax=axes[1], shrink=0.7, label="°")
    
    # 3) Green/Yellow Zone
    zone_map = np.where(slope_deg < SLOPE_THRESHOLD, 0, 1)
    cmap = ListedColormap(["#2ecc71", "#f39c12"])  # green, yellow
    im3 = axes[2].imshow(zone_map, cmap=cmap,
                          extent=[clip_lon_min, clip_lon_max, clip_lat_min, clip_lat_max])
    axes[2].set_title("로봇 배송 Zone (Green=가능, Yellow=제약)")
    
    # 성남시 경계 오버레이
    for ax in axes:
        seongnam.boundary.plot(ax=ax, color="black", linewidth=0.5)
    
    plt.tight_layout()
    plt.show()

In [5]:
if dem is not None:
    # GeoTIFF 저장
    clip_transform = rasterio.transform.from_bounds(
        clip_lon_min, clip_lat_min, clip_lon_max, clip_lat_max,
        dem.shape[1], dem.shape[0]
    )
    
    # DEM 저장
    with rasterio.open(
        OUT / "seongnam_dem.tif", "w", driver="GTiff",
        height=dem.shape[0], width=dem.shape[1], count=1,
        dtype="float32", crs="EPSG:4326", transform=clip_transform,
    ) as dst:
        dst.write(dem, 1)
    
    # 경사도 저장
    with rasterio.open(
        OUT / "seongnam_slope.tif", "w", driver="GTiff",
        height=slope_deg.shape[0], width=slope_deg.shape[1], count=1,
        dtype="float32", crs="EPSG:4326", transform=clip_transform,
    ) as dst:
        dst.write(slope_deg, 1)
    
    print(f"저장 완료:")
    print(f"  - {OUT / 'seongnam_dem.tif'}")
    print(f"  - {OUT / 'seongnam_slope.tif'}")

    # 동별 평균 경사도 계산 (Tableau 조인용 CSV 출력)
    from rasterio.features import geometry_mask
    
    dong_slopes = []
    for _, row in seongnam.iterrows():
        mask = geometry_mask(
            [row.geometry], out_shape=slope_deg.shape,
            transform=clip_transform, invert=True
        )
        masked_slope = slope_deg[mask]
        if len(masked_slope) > 0:
            dong_slopes.append({
                "SHP_ADM_CD": row["SHP_ADM_CD"],
                "ADM_NM": row["ADM_NM"],
                "avg_slope": np.nanmean(masked_slope),
                "max_slope": np.nanmax(masked_slope),
                "green_pct": np.nanmean(masked_slope < SLOPE_THRESHOLD) * 100,
            })
    
    df_slopes = pd.DataFrame(dong_slopes)
    df_slopes.to_csv(OUT / "dong_slope_stats.csv", index=False, encoding="utf-8-sig")
    print(f"  - {OUT / 'dong_slope_stats.csv'} ({len(df_slopes)} dongs)")
    print(f"\n경사 심한 동 Top 5:")
    print(df_slopes.nlargest(5, "avg_slope")[["ADM_NM", "avg_slope", "green_pct"]])